In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import torch
import shap

from transformers import AutoTokenizer, AutoModelForSequenceClassification


PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
TABLES_DIR = os.path.join(OUTPUTS_DIR, "tables")
FIGURES_DIR = os.path.join(OUTPUTS_DIR, "figures")

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

SHAP_DIR = os.path.join(FIGURES_DIR, "shap")
os.makedirs(SHAP_DIR, exist_ok=True)

FAKE_LABEL = 1
REAL_LABEL = 0
CLASS_NAMES = {REAL_LABEL: "Real(0)", FAKE_LABEL: "Fake(1)"}
OUTPUT_NAMES = [CLASS_NAMES[REAL_LABEL], CLASS_NAMES[FAKE_LABEL]]

MODEL_PATH = os.path.join(MODELS_DIR, "bert_fulltext_trunc256")
TEST_PATH = os.path.join(DATA_PROCESSED_DIR, "test.csv")

if not os.path.isdir(MODEL_PATH):
    raise FileNotFoundError(f"Model folder not found: {MODEL_PATH}")

if not os.path.isfile(TEST_PATH):
    raise FileNotFoundError(f"Test split not found: {TEST_PATH}")

required_any = {"config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json"}
present = set(os.listdir(MODEL_PATH))
if len(required_any.intersection(present)) < 2:
    raise FileNotFoundError(f"Model folder looks incomplete: {MODEL_PATH}\nFound files: {sorted(list(present))}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

test_df = pd.read_csv(TEST_PATH, usecols=["full_text", "label"]).copy()
test_df["full_text"] = test_df["full_text"].fillna("").astype(str)
test_df["label"] = test_df["label"].astype(int)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, local_files_only=True)
model.to(device)
model.eval()

MAX_LEN = 256
BATCH_SIZE = 16


def _normalize_texts(texts):
    if isinstance(texts, str):
        return [texts]
    if isinstance(texts, pd.Series):
        return ["" if x is None else str(x) for x in texts.tolist()]
    if isinstance(texts, np.ndarray):
        return ["" if x is None else str(x) for x in texts.ravel().tolist()]
    if isinstance(texts, (list, tuple)):
        flat = []
        for x in texts:
            if isinstance(x, (list, tuple, np.ndarray, pd.Series)):
                if isinstance(x, np.ndarray):
                    flat.extend(x.ravel().tolist())
                elif isinstance(x, pd.Series):
                    flat.extend(x.tolist())
                else:
                    flat.extend(list(x))
            else:
                flat.append(x)
        return ["" if x is None else str(x) for x in flat]
    return ["" if texts is None else str(texts)]


@torch.no_grad()
def predict_proba(texts):
    texts = _normalize_texts(texts)
    enc = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    logits = model(**enc).logits
    probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
    if probs.ndim != 2 or probs.shape[1] != 2:
        raise ValueError(f"Expected probs shape (N,2). Got: {probs.shape}")
    return probs


@torch.no_grad()
def infer_dataframe(df, text_col="full_text", label_col="label", batch_size=BATCH_SIZE):
    texts = df[text_col].fillna("").astype(str).tolist()
    y_true = df[label_col].astype(int).to_numpy()
    probs_all = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i + batch_size]
        probs_all.append(predict_proba(chunk))
    probs = np.vstack(probs_all)
    prob_fake = probs[:, FAKE_LABEL]
    y_pred = probs.argmax(axis=1).astype(int)
    out = df.copy()
    out["prob_fake(1)"] = prob_fake
    out["pred"] = y_pred
    out["correct"] = (y_pred == y_true).astype(int)
    return out


pred_df = infer_dataframe(test_df, batch_size=BATCH_SIZE)

fp_df = pred_df[(pred_df["label"] == REAL_LABEL) & (pred_df["pred"] == FAKE_LABEL)].copy()
fn_df = pred_df[(pred_df["label"] == FAKE_LABEL) & (pred_df["pred"] == REAL_LABEL)].copy()
tp_fake_df = pred_df[(pred_df["label"] == FAKE_LABEL) & (pred_df["pred"] == FAKE_LABEL)].copy()
tp_real_df = pred_df[(pred_df["label"] == REAL_LABEL) & (pred_df["pred"] == REAL_LABEL)].copy()

n_random_each = 3
n_conf_each = 3
n_err_each = 3

rng = 42

random_fake = pred_df[pred_df["label"] == FAKE_LABEL].sample(min(n_random_each, (pred_df["label"] == FAKE_LABEL).sum()), random_state=rng)
random_real = pred_df[pred_df["label"] == REAL_LABEL].sample(min(n_random_each, (pred_df["label"] == REAL_LABEL).sum()), random_state=rng)

conf_fake = tp_fake_df.sort_values("prob_fake(1)", ascending=False).head(n_conf_each)
conf_real = tp_real_df.sort_values("prob_fake(1)", ascending=True).head(n_conf_each)

err_fp = fp_df.sort_values("prob_fake(1)", ascending=False).head(n_err_each)
err_fn = fn_df.sort_values("prob_fake(1)", ascending=True).head(n_err_each)

selected = []

def _add_group(df_part, group):
    if df_part is None or len(df_part) == 0:
        return
    tmp = df_part.copy()
    tmp["group"] = group
    selected.append(tmp)

_add_group(random_fake, "random_true_fake")
_add_group(random_real, "random_true_real")
_add_group(conf_fake, "most_confident_fake")
_add_group(conf_real, "most_confident_real")
_add_group(err_fp, "errors_false_positive")
_add_group(err_fn, "errors_false_negative")

selected_df = pd.concat(selected, ignore_index=True)
selected_df = selected_df.reset_index(drop=True)

selected_out_path = os.path.join(TABLES_DIR, "shap_selected_examples.csv")
selected_df[["group", "label", "pred", "prob_fake(1)", "full_text"]].to_csv(selected_out_path, index=False)
print("Saved:", selected_out_path)
print("Selected examples:", len(selected_df))
print(selected_df["group"].value_counts())

masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(predict_proba, masker, output_names=OUTPUT_NAMES)

texts_for_shap = selected_df["full_text"].tolist()

max_evals = 256
batch_size = 8

shap_values = explainer(texts_for_shap, max_evals=max_evals, batch_size=batch_size)

token_rows = []

def _safe_filename(s):
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9_\-]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:80] if len(s) > 80 else s

for i in range(len(texts_for_shap)):
    group = selected_df.loc[i, "group"]
    true_y = int(selected_df.loc[i, "label"])
    pred_y = int(selected_df.loc[i, "pred"])
    prob_fake = float(selected_df.loc[i, "prob_fake(1)"])
    fname = f"{i:02d}_{_safe_filename(group)}_true{true_y}_pred{pred_y}_pf{prob_fake:.3f}.html"
    out_html = os.path.join(SHAP_DIR, fname)

    html_obj = shap.plots.text(shap_values[i], display=False)
    shap.save_html(out_html, html_obj)

    ex = shap_values[i]
    tokens = ex.data
    vals = ex.values
    if isinstance(vals, np.ndarray) and vals.ndim == 2 and vals.shape[1] == 2:
        contrib = vals[:, FAKE_LABEL]
    elif isinstance(vals, np.ndarray) and vals.ndim == 1:
        contrib = vals
    else:
        contrib = np.array(vals).reshape(-1)

    for j in range(len(tokens)):
        tok = tokens[j]
        sv = float(contrib[j]) if j < len(contrib) else np.nan
        token_rows.append({
            "example_id": i,
            "group": group,
            "true_label": true_y,
            "pred_label": pred_y,
            "prob_fake(1)": prob_fake,
            "token_pos": j,
            "token": str(tok),
            "shap_value_fake(1)": sv,
            "abs_shap_value_fake(1)": abs(sv) if not np.isnan(sv) else np.nan,
            "shap_html_path": out_html
        })

tokens_df = pd.DataFrame(token_rows)

topk = 25
top_tokens = (
    tokens_df.sort_values(["example_id", "abs_shap_value_fake(1)"], ascending=[True, False])
    .groupby("example_id", as_index=False)
    .head(topk)
    .reset_index(drop=True)
)

tokens_out_path = os.path.join(TABLES_DIR, "shap_top_tokens_per_example.csv")
top_tokens.to_csv(tokens_out_path, index=False)
print("Saved:", tokens_out_path)

print("Saved SHAP HTML files to:", SHAP_DIR)
print("Example file:", tokens_df["shap_html_path"].iloc[0] if len(tokens_df) else None)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import shap

TABLES_DIR = os.path.join(OUTPUTS_DIR, "tables")
FIGURES_DIR = os.path.join(OUTPUTS_DIR, "figures")
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

SHAP_DIR = os.path.join(FIGURES_DIR, "shap")
os.makedirs(SHAP_DIR, exist_ok=True)

def _safe_filename(s):
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9_\-]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:80] if len(s) > 80 else s

def _to_html(text_plot_obj):
    if hasattr(text_plot_obj, "data"):
        html = text_plot_obj.data
    else:
        html = text_plot_obj
    if not isinstance(html, str):
        html = str(html)
    if "<html" not in html.lower():
        html = "<html><head><meta charset='utf-8'></head><body>" + html + "</body></html>"
    return html

token_rows = []
topk = 25

for i in range(len(selected_df)):
    group = str(selected_df.loc[i, "group"])
    true_y = int(selected_df.loc[i, "label"])
    pred_y = int(selected_df.loc[i, "pred"])
    prob_fake = float(selected_df.loc[i, "prob_fake(1)"])

    fname = f"{i:02d}_{_safe_filename(group)}_true{true_y}_pred{pred_y}_pf{prob_fake:.3f}.html"
    out_html = os.path.join(SHAP_DIR, fname)

    html_obj = shap.plots.text(shap_values[i], display=False)
    html = _to_html(html_obj)
    with open(out_html, "w", encoding="utf-8") as f:
        f.write(html)

    ex = shap_values[i]
    tokens = ex.data
    vals = ex.values

    if isinstance(vals, np.ndarray) and vals.ndim == 2 and vals.shape[1] >= 2:
        contrib = vals[:, FAKE_LABEL]
    else:
        contrib = np.array(vals).reshape(-1)

    for j in range(len(tokens)):
        sv = float(contrib[j]) if j < len(contrib) else np.nan
        token_rows.append({
            "example_id": i,
            "group": group,
            "true_label": true_y,
            "pred_label": pred_y,
            "prob_fake(1)": prob_fake,
            "token_pos": j,
            "token": str(tokens[j]),
            "shap_value_fake(1)": sv,
            "abs_shap_value_fake(1)": abs(sv) if not np.isnan(sv) else np.nan,
            "shap_html_path": out_html
        })

tokens_df = pd.DataFrame(token_rows)

top_tokens = (
    tokens_df.sort_values(["example_id", "abs_shap_value_fake(1)"], ascending=[True, False])
    .groupby("example_id", as_index=False)
    .head(topk)
    .reset_index(drop=True)
)

tokens_out_path = os.path.join(TABLES_DIR, "shap_top_tokens_per_example.csv")
top_tokens.to_csv(tokens_out_path, index=False)

print("Saved SHAP HTML folder:", SHAP_DIR)
print("Saved:", tokens_out_path)
print("Example HTML:", top_tokens["shap_html_path"].iloc[0] if len(top_tokens) else None)

In [ ]:
import os
import re
import numpy as np
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
TABLES_DIR = os.path.join(OUTPUTS_DIR, "tables")
FIGURES_DIR = os.path.join(OUTPUTS_DIR, "figures")
SHAP_DIR = os.path.join(FIGURES_DIR, "shap")

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(SHAP_DIR, exist_ok=True)

FAKE_LABEL = 1

def merge_wordpieces(tokens, values):
    out_toks = []
    out_vals = []
    for t, v in zip(tokens, values):
        t = str(t)
        v = float(v)
        if t.startswith("##") and len(out_toks) > 0:
            out_toks[-1] = out_toks[-1] + t[2:]
            out_vals[-1] = out_vals[-1] + v
        else:
            out_toks.append(t)
            out_vals.append(v)
    return out_toks, out_vals

def is_junk_token(t):
    t = str(t).strip()
    if t == "":
        return True
    if t in ["[CLS]", "[SEP]", "[PAD]"]:
        return True
    if re.fullmatch(r"[\W_]+", t) is not None:
        return True
    return False

rows_all = []
rows_merged = []

for i in range(len(selected_df)):
    group = str(selected_df.loc[i, "group"])
    true_y = int(selected_df.loc[i, "label"])
    pred_y = int(selected_df.loc[i, "pred"])
    prob_fake = float(selected_df.loc[i, "prob_fake(1)"])

    ex = shap_values[i]
    tokens = list(ex.data)
    vals = ex.values

    if isinstance(vals, np.ndarray) and vals.ndim == 2 and vals.shape[1] >= 2:
        contrib = vals[:, FAKE_LABEL]
    else:
        contrib = np.array(vals).reshape(-1)

    for j, (t, sv) in enumerate(zip(tokens, contrib)):
        sv = float(sv)
        rows_all.append({
            "example_id": i,
            "group": group,
            "true_label": true_y,
            "pred_label": pred_y,
            "prob_fake(1)": prob_fake,
            "token_pos": j,
            "token": str(t),
            "shap_value_fake(1)": sv,
            "abs_shap_value_fake(1)": abs(sv)
        })

    m_tokens, m_vals = merge_wordpieces(tokens, contrib)
    for j, (t, sv) in enumerate(zip(m_tokens, m_vals)):
        sv = float(sv)
        rows_merged.append({
            "example_id": i,
            "group": group,
            "true_label": true_y,
            "pred_label": pred_y,
            "prob_fake(1)": prob_fake,
            "token_pos": j,
            "token": str(t),
            "shap_value_fake(1)": sv,
            "abs_shap_value_fake(1)": abs(sv),
            "is_junk": int(is_junk_token(t))
        })

tokens_all_df = pd.DataFrame(rows_all)
tokens_merged_df = pd.DataFrame(rows_merged)

all_out = os.path.join(TABLES_DIR, "shap_all_tokens_per_example.csv")
merged_out = os.path.join(TABLES_DIR, "shap_merged_tokens_per_example.csv")

tokens_all_df.to_csv(all_out, index=False, encoding="utf-8-sig")
tokens_merged_df.to_csv(merged_out, index=False, encoding="utf-8-sig")

topk = 25
top_merged = (
    tokens_merged_df[tokens_merged_df["is_junk"] == 0]
    .sort_values(["example_id", "abs_shap_value_fake(1)"], ascending=[True, False])
    .groupby("example_id", as_index=False)
    .head(topk)
    .reset_index(drop=True)
)

top_merged_out = os.path.join(TABLES_DIR, "shap_top_merged_tokens_per_example.csv")
top_merged.to_csv(top_merged_out, index=False, encoding="utf-8-sig")

group_top = (
    top_merged.groupby(["group", "token"], as_index=False)
    .agg(
        mean_abs_shap=("abs_shap_value_fake(1)", "mean"),
        mean_shap=("shap_value_fake(1)", "mean"),
        n=("token", "count")
    )
    .sort_values(["group", "mean_abs_shap"], ascending=[True, False])
)

group_top_out = os.path.join(TABLES_DIR, "shap_group_top_tokens.csv")
group_top.to_csv(group_top_out, index=False, encoding="utf-8-sig")

print("Saved:", all_out)
print("Saved:", merged_out)
print("Saved:", top_merged_out)
print("Saved:", group_top_out)
print(group_top.groupby("group").head(10))

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import shap
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
TABLES_DIR = os.path.join(OUTPUTS_DIR, "tables")
FIGURES_DIR = os.path.join(OUTPUTS_DIR, "figures")
SHAP_DIR = os.path.join(FIGURES_DIR, "shap")

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(SHAP_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

FAKE_LABEL = 1
REAL_LABEL = 0
LABEL_ORDER = [FAKE_LABEL, REAL_LABEL]
LABEL_TICKS = ["Fake(1)", "Real(0)"]

MAX_LEN = 256
BATCH_SIZE_INF = 16

best_dir = os.path.join(MODELS_DIR, "bert_fulltext_trunc256")
test_path = os.path.join(DATA_PROCESSED_DIR, "test.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

test_df = pd.read_csv(test_path, usecols=["full_text", "label"])
test_df["full_text"] = test_df["full_text"].fillna("").astype(str)
test_df["label"] = test_df["label"].astype(int)

tokenizer = AutoTokenizer.from_pretrained(best_dir, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(best_dir)
model.to(device)
model.eval()

class InferDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = [str(t) for t in texts]
        self.labels = [int(y) for y in labels]
        self.tokenizer = tokenizer
        self.max_len = int(max_len)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

infer_loader = DataLoader(
    InferDataset(test_df["full_text"].tolist(), test_df["label"].tolist(), tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE_INF,
    shuffle=False,
    num_workers=0
)

all_probs = []
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(infer_loader, desc="BERT inference"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].cpu().numpy()

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        preds = np.argmax(probs, axis=1)

        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(labels)

all_probs = np.concatenate(all_probs, axis=0)
all_preds = np.concatenate(all_preds, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

test_df["pred"] = all_preds
test_df["prob_fake(1)"] = all_probs[:, FAKE_LABEL]
test_df["prob_real(0)"] = all_probs[:, REAL_LABEL]

correct_fake = test_df[(test_df["label"] == FAKE_LABEL) & (test_df["pred"] == FAKE_LABEL)].copy()
correct_real = test_df[(test_df["label"] == REAL_LABEL) & (test_df["pred"] == REAL_LABEL)].copy()
fp = test_df[(test_df["label"] == REAL_LABEL) & (test_df["pred"] == FAKE_LABEL)].copy()
fn = test_df[(test_df["label"] == FAKE_LABEL) & (test_df["pred"] == REAL_LABEL)].copy()

rand_true_fake = correct_fake.sample(min(3, len(correct_fake)), random_state=SEED) if len(correct_fake) > 0 else correct_fake.head(0)
rand_true_real = correct_real.sample(min(3, len(correct_real)), random_state=SEED) if len(correct_real) > 0 else correct_real.head(0)

most_conf_fake = correct_fake.sort_values("prob_fake(1)", ascending=False).head(3)
most_conf_real = correct_real.sort_values("prob_fake(1)", ascending=True).head(3)

err_fp = fp.sort_values("prob_fake(1)", ascending=False).head(3)
err_fn = fn.sort_values("prob_fake(1)", ascending=True).head(3)

selected = []
selected.append(rand_true_fake.assign(group="random_true_fake"))
selected.append(rand_true_real.assign(group="random_true_real"))
selected.append(most_conf_fake.assign(group="most_confident_fake"))
selected.append(most_conf_real.assign(group="most_confident_real"))
selected.append(err_fp.assign(group="errors_false_positive"))
selected.append(err_fn.assign(group="errors_false_negative"))

selected_df = pd.concat(selected, ignore_index=True).reset_index(drop=True)
selected_df.insert(0, "example_id", np.arange(len(selected_df), dtype=int))

sel_csv_path = os.path.join(TABLES_DIR, "shap_selected_examples.csv")
selected_df.to_csv(sel_csv_path, index=False, encoding="utf-8-sig")
print("Saved:", sel_csv_path)
print("Selected examples:", len(selected_df))
print(selected_df["group"].value_counts())

def predict_fn(texts):
    if isinstance(texts, (pd.Series, np.ndarray, tuple, list)):
        texts = list(texts)
    else:
        texts = [texts]
    texts = ["" if t is None else str(t) for t in texts]
    enc = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
    return probs

masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(predict_fn, masker, output_names=["Real(0)", "Fake(1)"])

texts = selected_df["full_text"].tolist()
shap_values = explainer(texts)

def safe_filename(s):
    s = str(s)
    s = re.sub(r"[^\w\-_\.]+", "_", s, flags=re.UNICODE)
    return s[:160]

def html_from_shap_text_plot(expl):
    obj = shap.plots.text(expl, display=False)
    if hasattr(obj, "data") and isinstance(obj.data, str):
        return obj.data
    if hasattr(obj, "_repr_html_"):
        return obj._repr_html_()
    return str(obj)

rows = []
for i in range(len(selected_df)):
    ex_meta = selected_df.loc[i]
    group = str(ex_meta["group"])
    true_y = int(ex_meta["label"])
    pred_y = int(ex_meta["pred"])
    prob_fake = float(ex_meta["prob_fake(1)"])

    fname = f"{i:02d}_{group}_true{true_y}_pred{pred_y}_pf{prob_fake:.3f}.html"
    fname = safe_filename(fname)
    out_html = os.path.join(SHAP_DIR, fname)

    html_str = html_from_shap_text_plot(shap_values[i])
    with open(out_html, "w", encoding="utf-8") as f:
        f.write(html_str)

    ex = shap_values[i]
    tokens = list(ex.data)
    vals = ex.values

    if isinstance(vals, np.ndarray) and vals.ndim == 2 and vals.shape[1] >= 2:
        contrib = vals[:, FAKE_LABEL]
    else:
        contrib = np.array(vals).reshape(-1)

    for j, (t, sv) in enumerate(zip(tokens, contrib)):
        sv = float(sv)
        rows.append({
            "example_id": i,
            "group": group,
            "true_label": true_y,
            "pred_label": pred_y,
            "prob_fake(1)": prob_fake,
            "token_pos": j,
            "token": str(t),
            "shap_value_fake(1)": sv,
            "abs_shap_value_fake(1)": abs(sv),
            "shap_html_path": out_html
        })

tokens_df = pd.DataFrame(rows)
top_tokens_df = (
    tokens_df.sort_values(["example_id", "abs_shap_value_fake(1)"], ascending=[True, False])
    .groupby("example_id", as_index=False)
    .head(10)
    .reset_index(drop=True)
)

top_tokens_path = os.path.join(TABLES_DIR, "shap_top_tokens_per_example.csv")
top_tokens_df.to_csv(top_tokens_path, index=False, encoding="utf-8-sig")
print("Saved:", top_tokens_path)

def merge_wordpieces(tokens, values):
    out_toks = []
    out_vals = []
    for t, v in zip(tokens, values):
        t = str(t)
        v = float(v)
        if t.startswith("##") and len(out_toks) > 0:
            out_toks[-1] = out_toks[-1] + t[2:]
            out_vals[-1] = out_vals[-1] + v
        else:
            out_toks.append(t)
            out_vals.append(v)
    return out_toks, out_vals

def is_junk_token(t):
    t = str(t).strip()
    if t == "":
        return True
    if t in ["[CLS]", "[SEP]", "[PAD]"]:
        return True
    if re.fullmatch(r"[\W_]+", t) is not None:
        return True
    return False

merged_rows = []
for i in range(len(selected_df)):
    ex_meta = selected_df.loc[i]
    group = str(ex_meta["group"])
    true_y = int(ex_meta["label"])
    pred_y = int(ex_meta["pred"])
    prob_fake = float(ex_meta["prob_fake(1)"])
    ex = shap_values[i]
    tokens = list(ex.data)
    vals = ex.values
    if isinstance(vals, np.ndarray) and vals.ndim == 2 and vals.shape[1] >= 2:
        contrib = vals[:, FAKE_LABEL]
    else:
        contrib = np.array(vals).reshape(-1)
    m_tokens, m_vals = merge_wordpieces(tokens, contrib)
    for j, (t, sv) in enumerate(zip(m_tokens, m_vals)):
        sv = float(sv)
        merged_rows.append({
            "example_id": i,
            "group": group,
            "true_label": true_y,
            "pred_label": pred_y,
            "prob_fake(1)": prob_fake,
            "token_pos": j,
            "token": str(t),
            "shap_value_fake(1)": sv,
            "abs_shap_value_fake(1)": abs(sv),
            "is_junk": int(is_junk_token(t))
        })

merged_df = pd.DataFrame(merged_rows)
merged_path = os.path.join(TABLES_DIR, "shap_merged_tokens_per_example.csv")
merged_df.to_csv(merged_path, index=False, encoding="utf-8-sig")
print("Saved:", merged_path)

group_top = (
    merged_df[merged_df["is_junk"] == 0]
    .groupby(["group", "token"], as_index=False)
    .agg(
        mean_abs_shap=("abs_shap_value_fake(1)", "mean"),
        mean_shap=("shap_value_fake(1)", "mean"),
        n=("token", "count")
    )
    .sort_values(["group", "mean_abs_shap"], ascending=[True, False])
    .reset_index(drop=True)
)

group_top_path = os.path.join(TABLES_DIR, "shap_group_top_tokens.csv")
group_top.to_csv(group_top_path, index=False, encoding="utf-8-sig")
print("Saved:", group_top_path)

paper_rows = []
cfg = {
    "most_confident_fake": "positive",
    "most_confident_real": "negative",
    "errors_false_positive": "positive",
    "errors_false_negative": "negative",
    "random_true_fake": "positive",
    "random_true_real": "negative"
}

for g, direction in cfg.items():
    gdf = group_top[group_top["group"] == g].copy()
    if direction == "positive":
        gdf = gdf[gdf["mean_shap"] > 0]
    else:
        gdf = gdf[gdf["mean_shap"] < 0]
    gdf = gdf.sort_values("mean_abs_shap", ascending=False).head(15)
    gdf.insert(0, "direction", direction)
    paper_rows.append(gdf)

paper_df = pd.concat(paper_rows, ignore_index=True)
paper_path = os.path.join(TABLES_DIR, "shap_group_top_tokens_top15.csv")
paper_df.to_csv(paper_path, index=False, encoding="utf-8-sig")
print("Saved:", paper_path)

def plot_group_tokens(group_name, direction, k=15):
    gdf = group_top[group_top["group"] == group_name].copy()
    if direction == "positive":
        gdf = gdf[gdf["mean_shap"] > 0]
        title = f"{group_name}: top tokens pushing Fake(1)"
    else:
        gdf = gdf[gdf["mean_shap"] < 0]
        title = f"{group_name}: top tokens pushing Real(0)"
    gdf = gdf.sort_values("mean_abs_shap", ascending=False).head(k)
    if len(gdf) == 0:
        return
    gdf = gdf.sort_values("mean_shap", ascending=True)
    plt.figure(figsize=(9, 6))
    plt.barh(gdf["token"].tolist(), gdf["mean_shap"].tolist())
    plt.axvline(0.0, linewidth=1)
    plt.title(title)
    plt.xlabel("Mean SHAP contribution to Fake(1) probability")
    plt.ylabel("Token")
    plt.tight_layout()
    out_base = os.path.join(SHAP_DIR, safe_filename(f"shap_group_{group_name}_{direction}_top{k}"))
    plt.savefig(out_base + ".png", dpi=250, bbox_inches="tight")
    plt.savefig(out_base + ".pdf", bbox_inches="tight")
    plt.show()
    print("Saved:", out_base + ".png")
    print("Saved:", out_base + ".pdf")

for g, direction in cfg.items():
    plot_group_tokens(g, direction, k=15)